# Tutorial 09 - Selection methods for generative and predictive models

By the end of this section, students will be able to:

- Give examples of questions that can be answered by generative models and others that can be answered by predictive models.
- Discuss how the research question being asked impacts the statistical modelling procedures.
- Explain how regularized methods, such as lasso and ridge, can be used to estimate a predictive or a generative model.
- Distinguish the selection properties of lasso and ridge penalties.
- Discuss why the model obtained directly from lasso is not the most suitable model for generative modelling and how post-lasso is one way to address this problem.
- Write a computer script to perform lasso/ridge and use it to predict new outcomes.
- Write a computer script to perform post-lasso and use it to estimate a generative model.
- Discuss post-inference problems (e.g., double dipping into the data set) and current practical solutions available to address these (e.g., data-splitting techniques).
- Write a computer script to apply currently available practical solutions to post inference problems.
- Discuss how the research question being asked impacts the communication of the results.

In [ ]:
# Loading packages
library(car)
library(tidyverse)
library(tidymodels)
library(glmnet)
library(modeldata)
library(mltools)
source("tests_tutorial_09.R")


## Model Selection: Ridge

We have learned that *shrinkage methods* can be used to build predictive models. Some of these methods can also be used to select variables.

In particular, *Ridge method* will not select variables since the estimated coefficients won't be shrunk to zero. However, it can be used when

- there are more predictors than observations
- to address multicollinearity (in fact, that was the primary goal)

In this tutorial, you will use the dataset `concrete` from the `modeldata` package to build a Ridge regression and use it to predict the compressive strength of concrete in a test set.

**Data**: This dataset contains measurements on concrete mixtures, including the amounts of several ingredients and the age of the sample. The data originate from the Concrete Compressive Strength dataset compiled by Yeh (1998) and distributed through the UCI Machine Learning Repository. 

You will use `strength` as the response variable and the remaining 8 continuous variables as predictors. 

> Yeh I., "Modeling of strength of high performance concrete using artificial 
neural networks," *Cement and Concrete Research*, **28**, pp. 1797-1808 (1998)

*Run the code below to create the working dataset called `concrete_sample`*

The response variable `strength` is the measured compressive strength of concrete in megapascals (MPa) for a given mixture under a specific age (days).

The 8 input variables are:

- `cement`: amount of cement in the mixture (kg).
- `slag`: amount of blast furnace slag (kg).
- `fly_ash`: amount of fly ash (kg).
- `water`: amount of water (kg).
- `superplasticizer`: amount of superplasticizer (kg).
- `coarse_agg`: amount of coarse aggregate (kg).
- `fine_agg`: amount of fine aggregate (kg).
- `age`: age of the sample in days (days).

Ridge regression may be particularly useful to analyze the data since mixture components tend to be correlated.

In [ ]:
# Load and clean the concrete data

data("concrete", package = "modeldata")

concrete_sample <- concrete

names(concrete_sample) <- c(
  "cement", "slag", "fly_ash", "water", "superplasticizer",
  "coarse_agg", "fine_agg", "age", "strength"
)

# Split data into training and test sets
set.seed(123)

training_concrete <- concrete_sample %>%
  sample_frac(0.6)

testing_concrete <- concrete_sample %>%
  setdiff(training_concrete)

# Build matrix and vector required by `glmnet`
# Using `model.matrix()` to get the X matrix; the first column
# corresponds to the intercept and needs to be deleted

concrete_X_train <- model.matrix(object = strength ~ .,
  data = training_concrete)[, -1]
concrete_Y_train <- as.matrix(training_concrete[, "strength"])

concrete_X_test <- model.matrix(object = strength ~ .,
  data = testing_concrete)[, -1]

concrete_Y_test <- as.matrix(testing_concrete[, "strength"])


**Question 1.0**
<br>{points: 1}

Now that we have our training data prepared in `concrete_X_train` and `concrete_Y_train`, we will select the value of $\lambda$ using cross-validation and the one-standard-error (1SE) rule.

We can do this automatically with function `cv.glmnet()` where `x` is the matrix of input variables and `y` is the vector of responses from the training set that we prepared.

> **Recall**: the method of Ridge regression is defined when `alpha = 0`.

To select `lambda` we will use a **sequence** of values that goes from $\lambda = \exp(-5) = 0.0067$ to $\lambda = \exp(10) = 22026.5$. This is an optional argument. If not specified, `glmnet()` will generate an appropriate sequence for us.

The function `cv.glmnet()` uses cross-validation to compare the average (across folds) test MSE at each of the $\lambda$ values in the grid. 

Assign the function's output as `concrete_cv_lambda_ridge`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
set.seed(1234) # DO NOT CHANGE!

# concrete_cv_lambda_ridge <- ...(
#   x = ..., y = ...,
#   alpha = ...,
#   lambda = exp(seq(-5, 10, 0.1))
# )

# your code here
fail() # No Answer - remove if you provide an answer


In [ ]:
test_1.0()

**Question 1.1**
<br>{points: 1}

We can visualize the estimated test MSE at each value of lambda in the sequence by using `concrete_cv_lambda_ridge` and `plot()`.

The resulting plot will indicate the $\text{MSE}_{\text{test}}$ on the $y$-axis (error bars show the variation of the test error in the different folds) along with the range of $\lambda$ on the bottom $x$-axis on the natural log-scale.

> **Recall**: Ridge regression never shrinks estimators to zero, so the top axis will continue to show all 8 predictors in the model.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
# plot_data <- ...

# your code here
fail() # No Answer - remove if you provide an answer

plot(plot_data, main = "MSE of Ridge estimated by CV for different lambdas

")


In [ ]:
test_1.1()

**Question 1.2**
<br>{points: 1}

The plot in **Question 1.1** also shows two vertical dotted lines. *Given an `object` coming from `cv.glmnet()`*, these lines correspond to two values of $\lambda$:

- $\hat{\lambda}_{\text{min}}$ which minimizes MSE. It can be obtained with `object$lambda.min`.

- $\hat{\lambda}_{\text{1SE}}$ the largest value for which the CV-MSE is within one standard error of the minimum. It can be obtained with `object$lambda.1se`.

Using `concrete_cv_lambda_ridge`, obtain $\hat{\lambda}_{\text{1SE}}$ and save it as `concrete_lambda_1se_MSE_ridge`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
# concrete_lambda_1se_MSE_ridge <- round(..., 4)

# your code here
fail() # No Answer - remove if you provide an answer

concrete_lambda_1se_MSE_ridge


In [ ]:
test_1.2()

**Question 1.3**
<br>{points: 1}

Once we have selected a value of $\lambda$, we can extract the estimated Ridge regression at that level of penalization. Store the estimated coefficients in `concrete_ridge_1se_coef`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
set.seed(1234) # DO NOT CHANGE!

# concrete_ridge_1se_coef <- ...(..., s = ...)

# your code here
fail() # No Answer - remove if you provide an answer

concrete_ridge_1se_coef


In [ ]:
test_1.3()

**Question 1.4**
<br>{points: 1}

Let's compare the estimated regression coefficients of `concrete_ridge_1se_coef` with those of `concrete_full_OLS`. Both models are estimated using the training data only.

Create a data frame called `concrete_coefs` with two columns:

- `Full_OLS`: The estimated coefficients from `concrete_full_OLS` obtained via function `coef()`.
- `Ridge_1SE`: The estimated coefficients in `concrete_ridge_1se_coef`. Recall this is the estimated ridge regression with $\hat{\lambda}_{\text{1SE}}$.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# concrete_full_OLS <- ...

# concrete_coefs <- cbind(
#   Full_OLS = ...(...),
#   Ridge_1SE = as.vector(...)) %>%
#       round(4) %>% as.data.frame()

# your code here
fail() # No Answer - remove if you provide an answer

concrete_coefs


In [ ]:
test_1.4()

**Question 1.5**
<br>{points: 1}

We can also visualize how the estimated Ridge regression coefficients change with the level of penalization. Using `concrete_cv_lambda_ridge$glmnet.fit`, plot the coefficient paths against $\log(\lambda)$.

Store the fitted model in `concrete_ridge_coefficient_path` and use `plot()` with `xvar = "lambda"`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
# concrete_ridge_coefficient_path <- ...
# plot(..., xvar = "lambda")

# your code here
fail() # No Answer - remove if you provide an answer


In [ ]:
test_1.5()


**Question 1.6**
<br>{points: 1}

Based on the coefficient-path plot in **Question 1.5**, is the following statement **True or False**?

*As $\lambda$ increases, Ridge regression shrinks predictor coefficients toward zero, but does not set them exactly equal to zero.*

Store your answer as `"true"` or `"false"` in `answer1.6`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
# answer1.6 <- "..."

# your code here
fail() # No Answer - remove if you provide an answer


In [ ]:
test_1.6()


**Question 1.7**
<br>{points: 1}

Write code following the steps outlined below.

- Using `predict()` and `concrete_full_OLS`, obtain the out-of-sample predicted `strength` values for the mixtures in `testing_concrete`. Store them in a variable called `concrete_test_pred_full_OLS`.

Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.


In [ ]:
# concrete_test_pred_full_OLS <- ...

# your code here
fail() # No Answer - remove if you provide an answer

head(concrete_test_pred_full_OLS)


In [ ]:
test_1.7()

**Question 1.8**
<br>{points: 1}

We will now compute the **Mean Squared Error (MSE)** on the test set to evaluate the predictive model (the smaller the metric, the better the model):

> **Heads up:** a related measure commonly used is the **Root Mean Squared Error (RMSE) = $\sqrt{\text{MSE}}$**, which measures the typical magnitude of the prediction errors. This metric has the same units as the response.

Use the function `rmse()` from the `mltools` package to compute the $\text{RMSE}_{\text{test}}$ of the predicted concrete strengths in `concrete_test_pred_full_OLS`.

Store the computed RMSE metric in a tibble called `concrete_test_RMSEs` with two columns:

- `Model`: The regression model from which we obtain the prediction accuracy.
- `RMSE`: The corresponding $\text{RMSE}_{\text{test}}$.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# concrete_test_RMSEs <- tibble(
#   Model = "OLS Full Regression",
#   RMSE = ...(
#     preds = ...,
#     actuals = ...
#   )
# )

# your code here
fail() # No Answer - remove if you provide an answer

concrete_test_RMSEs

In [ ]:
test_1.8()

**Question 1.9**
<br>{points: 1}

Use `concrete_cv_lambda_ridge` and the value of $\lambda$ selected by the one-standard-error (1SE) rule to predict the concrete strength in the test set `testing_concrete`, and call the resulting object `concrete_test_pred_ridge_1se`.

> **Hint:** Use function `predict()` with the argument `newx` to specify the test set.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
# concrete_test_pred_ridge_1se <- predict(...,
#   newx = ..., ...)

# your code here
fail() # No Answer - remove if you provide an answer


In [ ]:
test_1.9()

**Question 1.10**
<br>{points: 1}

Use the function `rmse()` to compute $\text{RMSE}_{\text{test}}$ using the predicted values stored in `concrete_test_pred_ridge_1se`.

Add this metric as an additional row in the tibble `concrete_test_RMSEs`. Use `"Ridge Regression with 1SE rule"` in column `Model` and the corresponding value for $\text{RMSE}_{\text{test}}$ in column `RMSE`.

**NOTE**: the code below binds rows into `concrete_test_RMSEs`. The code below adds a row to concrete_test_RMSEs. Run this cell only once; otherwise, the object will contain duplicate rows.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*


In [ ]:
# concrete_test_RMSEs <- rbind(
#   concrete_test_RMSEs,
#   tibble(
#     Model = "Ridge Regression with 1SE rule",
#     RMSE = ...(
#       ...,
#       ...
#     )
#   )
# )

# your code here
fail() # No Answer - remove if you provide an answer

concrete_test_RMSEs


In [ ]:
test_1.10()

The model with the smaller test RMSE shows the best predictive performance. It is reasonable to expect Ridge to do better than Least Squares because the penalty was tuned by cross-validation and the concrete mixture variables can be moderately correlated. However, results may differ under a different partition of the data.